# Feature Engineering for Mixed-Frequency ML Nowcasting

**Module 05 — Machine Learning Extensions**

**Learning objectives:**
- Compare three feature engineering strategies: flat stacking, statistical summaries, PCA embeddings
- Measure information retention at each step
- Evaluate dimensionality vs predictive performance trade-offs
- Apply each strategy to daily, monthly, and quarterly mixed-frequency data

**Estimated time**: 12 minutes  
**Data**: Synthetic mixed-frequency dataset (daily, monthly, quarterly)

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

In [ ]:
section_divider("1. Generate Multi-Frequency Data")

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Generate Multi-Frequency Data

We create a realistic three-frequency dataset:
- **Daily**: 5 financial indicators (stock returns, spreads, commodity prices)
- **Monthly**: 6 macro indicators (ISM, payrolls, housing starts, etc.)
- **Quarterly target**: GDP growth

In [ ]:
def generate_multi_frequency_data(n_quarters=80, seed=42):
    """
    Generate realistic three-frequency dataset.
    Daily: 65 trading days/quarter, 5 indicators
    Monthly: 3 months/quarter, 6 indicators  
    Quarterly: target (GDP growth)
    """
    np.random.seed(seed)
    
    DAYS_PER_Q = 65   # Approx trading days per quarter
    MONTHS_PER_Q = 3
    
    N_DAILY = 5
    N_MONTHLY = 6
    
    total_days = n_quarters * DAYS_PER_Q
    total_months = n_quarters * MONTHS_PER_Q
    
    # Daily indicators (financial)
    daily_data = np.zeros((total_days, N_DAILY))
    rho_d = [0.98, 0.95, 0.92, 0.88, 0.85]
    for i, rho in enumerate(rho_d):
        for t in range(1, total_days):
            daily_data[t, i] = rho * daily_data[t-1, i] + np.sqrt(1-rho**2) * np.random.randn()
    
    # Monthly indicators (macro)
    monthly_data = np.zeros((total_months, N_MONTHLY))
    rho_m = [0.90, 0.85, 0.80, 0.75, 0.70, 0.65]
    for i, rho in enumerate(rho_m):
        for t in range(1, total_months):
            monthly_data[t, i] = rho * monthly_data[t-1, i] + np.sqrt(1-rho**2) * np.random.randn()
    
    # Common factor driving all frequencies
    common = np.sin(2 * np.pi * np.arange(n_quarters) / 16)  # 4-year cycle
    
    # Quarterly GDP growth
    gdp = np.zeros(n_quarters)
    for q in range(n_quarters):
        d_start = q * DAYS_PER_Q
        m_start = q * MONTHS_PER_Q
        
        # Sum of daily indicator 0 (most recent week = last 5 days)
        daily_signal = daily_data[d_start+DAYS_PER_Q-5:d_start+DAYS_PER_Q, 0].mean()
        
        # Monthly indicator 1 change
        monthly_change = monthly_data[m_start + 2, 1] - monthly_data[m_start, 1]
        
        # Nonlinear: threshold effect when daily_signal < -1.5
        nonlinear = -2.0 * (daily_signal < -1.5)
        
        gdp[q] = 2.0 + 0.8 * common[q] + 0.4 * daily_signal + 0.3 * monthly_change + nonlinear + 0.3 * np.random.randn()
    
    daily_names = [f'daily_{i}' for i in range(N_DAILY)]
    monthly_names = [f'monthly_{i}' for i in range(N_MONTHLY)]
    
    return {
        'daily': daily_data,
        'monthly': monthly_data,
        'quarterly_gdp': gdp,
        'n_quarters': n_quarters,
        'days_per_q': DAYS_PER_Q,
        'months_per_q': MONTHS_PER_Q,
        'daily_names': daily_names,
        'monthly_names': monthly_names,
    }


data = generate_multi_frequency_data(n_quarters=80)
print(f"Dataset dimensions:")
print(f"  Daily: {data['daily'].shape} ({data['n_quarters']} quarters × {data['days_per_q']} days × {data['daily'].shape[1]} series)")
print(f"  Monthly: {data['monthly'].shape} ({data['n_quarters']} quarters × {data['months_per_q']} months × {data['monthly'].shape[1]} series)")
print(f"  Quarterly GDP: {data['quarterly_gdp'].shape}")

In [ ]:
section_divider("2. Strategy A: Flat Stacking")

## 2. Strategy A: Flat Stacking

Concatenate all lags as individual features. For daily data with 65 lags per quarter and 5 series: 325 features. For monthly with 3 months and 6 series: 18 features. Total: 343 features.

In [ ]:
def flat_stack_features(data):
    """
    Strategy A: Flat stacking of all high-frequency lags.
    Returns feature matrix with all daily and monthly lags.
    """
    n_q = data['n_quarters']
    dpq = data['days_per_q']
    mpq = data['months_per_q']
    
    rows = []
    for q in range(1, n_q):  # Skip first quarter (need lags)
        row = []
        
        # Daily lags (65 lags × 5 series = 325 features)
        d_end = q * dpq
        daily_window = data['daily'][d_end-dpq:d_end]  # Current quarter's daily data
        for s in range(data['daily'].shape[1]):
            row.extend(daily_window[::-1, s])  # Most recent first
        
        # Monthly lags (3 lags × 6 series = 18 features)
        m_end = q * mpq
        monthly_window = data['monthly'][m_end-mpq:m_end]  # Current quarter's monthly data
        for s in range(data['monthly'].shape[1]):
            row.extend(monthly_window[::-1, s])  # Most recent first
        
        rows.append(row)
    
    # Feature names
    feat_names = []
    for s_name in data['daily_names']:
        for lag in range(1, dpq + 1):
            feat_names.append(f"{s_name}_L{lag}")
    for s_name in data['monthly_names']:
        for lag in range(1, mpq + 1):
            feat_names.append(f"{s_name}_L{lag}")
    
    X = np.array(rows)
    y = data['quarterly_gdp'][1:]
    
    return X, y, feat_names


X_flat, y, feat_names_flat = flat_stack_features(data)
print(f"Strategy A (Flat Stacking):")
print(f"  Observations: {X_flat.shape[0]}")
print(f"  Features: {X_flat.shape[1]}")
print(f"  Daily features: {data['days_per_q'] * data['daily'].shape[1]}")
print(f"  Monthly features: {data['months_per_q'] * data['monthly'].shape[1]}")

In [ ]:
section_divider("3. Strategy B: Statistical Summary Features")

## 3. Strategy B: Statistical Summary Features

In [ ]:
def compute_window_stats(window):
    """
    Compute summary statistics for a single series' lag window.
    Returns 8 statistics capturing level, trend, volatility, and extremes.
    """
    n = len(window)
    t_idx = np.arange(n)
    
    stats = {
        'mean': np.mean(window),
        'std': np.std(window),
        'last': window[-1],           # Most recent
        'first': window[0],           # Oldest
        'momentum': window[-1] - window[0],
        'acceleration': np.mean(np.diff(window[-min(5, n):]) ),  # Recent average change
        'max_drawup': np.max(window) - window[0],
        'trend_slope': np.polyfit(t_idx, window, 1)[0]
    }
    return list(stats.values()), list(stats.keys())


def summary_features(data):
    """
    Strategy B: Statistical summaries of lag windows.
    8 stats × 5 daily series + 8 stats × 6 monthly series = 88 features.
    """
    n_q = data['n_quarters']
    dpq = data['days_per_q']
    mpq = data['months_per_q']
    
    rows = []
    for q in range(1, n_q):
        row = []
        
        # Daily summaries
        d_end = q * dpq
        daily_window = data['daily'][d_end-dpq:d_end]
        for s in range(data['daily'].shape[1]):
            stats_vals, _ = compute_window_stats(daily_window[:, s])
            row.extend(stats_vals)
        
        # Monthly summaries
        m_end = q * mpq
        monthly_window = data['monthly'][m_end-mpq:m_end]
        for s in range(data['monthly'].shape[1]):
            stats_vals, _ = compute_window_stats(monthly_window[:, s])
            row.extend(stats_vals)
        
        rows.append(row)
    
    # Feature names
    stat_names = ['mean', 'std', 'last', 'first', 'momentum', 'accel', 'drawup', 'slope']
    feat_names = []
    for s_name in data['daily_names'] + data['monthly_names']:
        for stat in stat_names:
            feat_names.append(f"{s_name}_{stat}")
    
    X = np.array(rows)
    y = data['quarterly_gdp'][1:]
    
    return X, y, feat_names


X_summary, y, feat_names_summary = summary_features(data)
print(f"Strategy B (Statistical Summaries):")
print(f"  Features: {X_summary.shape[1]} (vs {X_flat.shape[1]} flat — {100 * X_summary.shape[1] / X_flat.shape[1]:.0f}% of original)")
print(f"  Reduction factor: {X_flat.shape[1] / X_summary.shape[1]:.1f}×")

In [ ]:
section_divider("4. Strategy C: PCA Embeddings")

## 4. Strategy C: PCA Embeddings

For daily data (65 lags per series), PCA compresses each series into d components capturing 90%+ of variance.

In [ ]:
def pca_embedded_features(data, n_daily_components=3, n_monthly_components=2):
    """
    Strategy C: PCA embedding of lag windows.
    Fits PCA on all quarterly windows, extracts components per quarter.
    """
    n_q = data['n_quarters']
    dpq = data['days_per_q']
    mpq = data['months_per_q']
    
    # Build lag matrix for all quarters (for PCA fitting)
    daily_windows = []
    monthly_windows = []
    
    for q in range(1, n_q):
        d_end = q * dpq
        daily_windows.append(data['daily'][d_end-dpq:d_end])  # (dpq, n_daily_series)
        m_end = q * mpq
        monthly_windows.append(data['monthly'][m_end-mpq:m_end])
    
    # Fit PCA on daily lag profiles: stack all windows, shape = (n_q * n_daily, dpq) per series
    pca_daily = []
    explained_daily = []
    for s in range(data['daily'].shape[1]):
        lag_mat = np.stack([w[:, s] for w in daily_windows])  # (n_q-1, dpq)
        pca = PCA(n_components=n_daily_components, random_state=42)
        pca.fit(lag_mat)
        pca_daily.append(pca)
        explained_daily.append(pca.explained_variance_ratio_.sum())
    
    pca_monthly = []
    explained_monthly = []
    for s in range(data['monthly'].shape[1]):
        lag_mat = np.stack([w[:, s] for w in monthly_windows])
        pca = PCA(n_components=min(n_monthly_components, mpq - 1), random_state=42)
        pca.fit(lag_mat)
        pca_monthly.append(pca)
        explained_monthly.append(pca.explained_variance_ratio_.sum())
    
    # Extract PCA features per quarter
    rows = []
    for q_idx, q in enumerate(range(1, n_q)):
        row = []
        
        for s in range(data['daily'].shape[1]):
            w = daily_windows[q_idx][:, s].reshape(1, -1)
            embedding = pca_daily[s].transform(w)[0]
            row.extend(embedding)
        
        for s in range(data['monthly'].shape[1]):
            w = monthly_windows[q_idx][:, s].reshape(1, -1)
            embedding = pca_monthly[s].transform(w)[0]
            row.extend(embedding)
        
        rows.append(row)
    
    # Feature names
    feat_names = []
    for s_name in data['daily_names']:
        for k in range(n_daily_components):
            feat_names.append(f"{s_name}_PC{k+1}")
    for s_name in data['monthly_names']:
        for k in range(min(n_monthly_components, mpq - 1)):
            feat_names.append(f"{s_name}_PC{k+1}")
    
    X = np.array(rows)
    y = data['quarterly_gdp'][1:]
    
    print(f"  Daily PCA: {n_daily_components} components per series")
    print(f"    Explained variance: {np.mean(explained_daily):.1%} (avg across series)")
    print(f"  Monthly PCA: {min(n_monthly_components, mpq-1)} components per series")
    print(f"    Explained variance: {np.mean(explained_monthly):.1%} (avg across series)")
    
    return X, y, feat_names


print("Strategy C (PCA Embeddings):")
X_pca, y, feat_names_pca = pca_embedded_features(data, n_daily_components=3, n_monthly_components=2)
print(f"  Features: {X_pca.shape[1]} (vs {X_flat.shape[1]} flat — {100 * X_pca.shape[1] / X_flat.shape[1]:.0f}% of original)")
print(f"  Reduction factor: {X_flat.shape[1] / X_pca.shape[1]:.0f}×")

In [ ]:
section_divider("5. Dimensionality vs Predictive Performance Comparison")

## 5. Dimensionality vs Predictive Performance Comparison

In [ ]:
def cv_rmse(X, y, n_splits=5, min_train=20):
    """
    Time-series CV RMSE for Random Forest.
    Returns mean RMSE across validation folds.
    """
    T = len(y)
    fold_rmses = []
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    sc = StandardScaler()
    
    for train_idx, val_idx in tscv.split(X):
        if len(train_idx) < min_train:
            continue
        
        X_tr = sc.fit_transform(X[train_idx])
        X_va = sc.transform(X[val_idx])
        y_tr = y[train_idx]
        y_va = y[val_idx]
        
        rf = RandomForestRegressor(
            n_estimators=100, min_samples_leaf=3,
            random_state=42, n_jobs=-1
        )
        rf.fit(X_tr, y_tr)
        rmse = np.sqrt(mean_squared_error(y_va, rf.predict(X_va)))
        fold_rmses.append(rmse)
    
    return np.mean(fold_rmses), np.std(fold_rmses)


print("Comparing strategies (Random Forest, 5-fold TS-CV):")
print(f"{'Strategy':<25} {'Features':<10} {'CV RMSE (±std)':<20}")
print("-" * 55)

strategies = [
    ('A: Flat Stacking', X_flat, feat_names_flat),
    ('B: Summary Stats', X_summary, feat_names_summary),
    ('C: PCA Embeddings', X_pca, feat_names_pca),
]

strategy_results = {}
for name, X, fnames in strategies:
    rmse_mean, rmse_std = cv_rmse(X, y)
    strategy_results[name] = {'rmse': rmse_mean, 'std': rmse_std, 'n_features': X.shape[1]}
    print(f"{name:<25} {X.shape[1]:<10} {rmse_mean:.4f} ± {rmse_std:.4f}")

## 6. Visualise Trade-offs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: scatter plot of features vs RMSE
ax = axes[0]
strategy_colors = {'A: Flat Stacking': 'red', 'B: Summary Stats': 'blue', 'C: PCA Embeddings': 'green'}

for name, res in strategy_results.items():
    ax.scatter(res['n_features'], res['rmse'],
               s=200, color=strategy_colors[name], zorder=5, label=name)
    ax.errorbar(res['n_features'], res['rmse'], yerr=res['std'],
                color=strategy_colors[name], fmt='none', linewidth=2)

ax.set_xlabel('Number of Features', fontsize=12)
ax.set_ylabel('CV RMSE', fontsize=12)
ax.set_title('Dimensionality vs Forecast Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: PCA explained variance for daily series
ax2 = axes[1]
n_q = data['n_quarters']
dpq = data['days_per_q']

daily_windows = []
for q in range(1, n_q):
    d_end = q * dpq
    daily_windows.append(data['daily'][d_end-dpq:d_end])

# Show PCA explained variance curve for first daily series
lag_mat = np.stack([w[:, 0] for w in daily_windows])
pca_full = PCA(n_components=min(20, dpq), random_state=42)
pca_full.fit(lag_mat)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

ax2.plot(range(1, len(cum_var)+1), cum_var, 'b-o', markersize=4)
ax2.axhline(0.90, color='red', linestyle='--', label='90% threshold')
ax2.axhline(0.95, color='orange', linestyle='--', label='95% threshold')
for k, (n_comp, thresh) in enumerate([(3, 0.90), (5, 0.95)]):
    idx = np.searchsorted(cum_var, thresh)
    ax2.axvline(idx + 1, color=['red', 'orange'][k], linestyle=':', alpha=0.7)
    ax2.annotate(f'{idx+1} components\n{thresh*100:.0f}%',
                xy=(idx+1, thresh), xytext=(idx+3, thresh - 0.05),
                fontsize=8)

ax2.set_xlabel('Number of PCA Components')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title('PCA: How Many Components for Daily Series?')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../resources/feature_engineering_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Hybrid: PCA Embeddings + Summary Statistics

Combine PCA embeddings (capturing variance structure) with summary statistics (capturing economically meaningful aggregates) for the best of both worlds.

In [ ]:
# Hybrid: concatenate PCA embeddings with summary statistics
X_hybrid = np.concatenate([X_pca, X_summary], axis=1)
feat_names_hybrid = feat_names_pca + feat_names_summary

print(f"Hybrid features: {X_hybrid.shape[1]}")
print(f"  = {X_pca.shape[1]} PCA + {X_summary.shape[1]} summaries")

rmse_hybrid, rmse_hybrid_std = cv_rmse(X_hybrid, y)
print(f"Hybrid CV RMSE: {rmse_hybrid:.4f} ± {rmse_hybrid_std:.4f}")

# Final comparison table
strategy_results['D: Hybrid (PCA+Stats)'] = {
    'rmse': rmse_hybrid, 'std': rmse_hybrid_std,
    'n_features': X_hybrid.shape[1]
}

print("\n=== Final Comparison ===")
print(f"{'Strategy':<30} {'Features':<10} {'CV RMSE':<12} {'Reduction':<10}")
print("-" * 62)
baseline_feats = X_flat.shape[1]
for name, res in strategy_results.items():
    reduction = baseline_feats / res['n_features']
    print(f"{name:<30} {res['n_features']:<10} {res['rmse']:<12.4f} {reduction:<10.1f}×")

## 8. Summary

Feature engineering for mixed-frequency ML nowcasting:

| Strategy | Features | Pros | Cons |
|----------|----------|------|------|
| Flat stacking | 343 | No information loss | Curse of dimensionality, correlated features |
| Summary stats | 88 | Interpretable, economically meaningful | May miss lag-specific effects |
| PCA embeddings | 27 | Maximum compression | Components lose economic meaning |
| Hybrid | 115 | Balance of information types | Slightly more complex |

**Key takeaways:**
- For macro data (T~100, monthly frequencies), summary statistics often match or beat flat stacking
- For daily data (65+ lags), PCA compression is essential — all variance concentrated in first few PCs
- The hybrid approach (PCA for daily, summaries for monthly) is a robust practical choice
- Always validate choices with time-series CV on your specific dataset

**Next**: `../exercises/01_ml_extensions_self_check.py` — self-check exercise.

In [ ]:
key_takeaways(["- For macro data (T~100, monthly frequencies), summary statistics often match or", "For daily data (65+ lags), PCA compression is essential — all variance concentra", "The hybrid approach (PCA for daily, summaries for monthly) is a robust practical", "Always validate choices with time-series CV on your specific dataset"])